# Reading Multiple Traces
---

In the previous notebook you read a single multi-GPU trace and learned to spot the NCCL kernel bars, NVLink metrics, and the symptoms each pattern points at.  Real-world performance work usually involves more than one trace though:

- **Comparing runs**: baseline vs. optimized, or different precisions, or different batch sizes.
- **Comparing ranks**: in a DDP run, GPU3 is slower than the others — is that a hardware issue, input skew, or just measurement noise?  You can't tell from one rank's data.
- **Detecting load imbalance**: are all ranks finishing the AllReduce at the same time, or is one consistently slower?

Nsight Systems gives you two complementary techniques for this:

1. **The GUI multi-report view** — visually align multiple `.nsys-rep` files on a single timeline.  Best for ad-hoc exploration.
2. **`nsys recipe`** — Python scripts that extract structured data from one or more reports.  Best for automated analysis, regression tracking, and answering precise quantitative questions ("what fraction of step time is in NCCL?").

## The Multi-Report View in the GUI

You can open any number of `.nsys-rep` files in a single Nsight Systems timeline:

- **File → Open…**, select multiple files in the dialog, *or*
- **File → New multi-report view…**, add files in the editor pane, click **Apply**.

<img src="images/new-multi-report-view.png" alt="multi-report view editor">

Each report appears as its own track in the merged timeline, sharing a common time axis.  This is the easiest way to *see* how one run differs from another:

<img src="images/multi-report-view.png" alt="multi-report view loaded">

Try opening `baseline_nvtx.nsys-rep` and `firstOptim.nsys-rep` from the previous notebook side by side.  The NVTX `data_load` ranges that were wide in the baseline collapse in the optimized run, and the GPU gaps fill in with kernel activity.

The multi-report view is great for "look at these two together," but it doesn't help when you want a *quantitative* answer — "what fraction of step time was in NCCL across all four ranks?" or "did optimization X actually overlap compute with communication?"  For those, `nsys recipe`.

## The `nsys recipe` System

`nsys recipe` runs Python scripts that pull structured data out of `.nsys-rep` files.  The recipes ship with Nsight Systems (typically under `<nsys-install>/python/packages/nsys-recipe/recipes/`) and you can write your own.

To list what's available:

In [ ]:
!nsys recipe --help

Recipes come in two flavors:

- **Single-report recipes** operate on one `.nsys-rep` and produce a summary (per-NVTX-range time, per-kernel statistics, etc.).  You can run these on any of the reports you already have from earlier in Lab 3.
- **Multi-report recipes** consume multiple `.nsys-rep` files and compare or aggregate across them.  These are what you need for cross-rank questions — and they require the *right shape* of input, which we'll come to in a moment.

> **Note**: recipe names and exact CLI flags can vary by Nsight Systems version.  Run `nsys recipe --help` and `nsys recipe <name> --help` to confirm the invocations on your container; the patterns below match the version shipped with the workshop image.

## Single-Report Recipe Example

Let's run a single-report recipe on the baseline DDP profile you already produced.  `nvtx_sum` aggregates time spent in each NVTX range — exactly what we want to see how the training step decomposed.

In [ ]:
!nsys recipe nvtx_sum --input /workspace/reports/baseline_nvtx.nsys-rep

The output is a table of NVTX range names with aggregate time, percentage, and instance count.  In the baseline trace, `data_load` should be the largest entry — that's exactly the bug `nsys-application.ipynb` had us identify and fix.

Run the same recipe against the optimized run to see the change:

In [ ]:
!nsys recipe nvtx_sum --input /workspace/reports/firstOptim.nsys-rep

`data_load` should now be a much smaller fraction.  The same information is visible in the GUI by hand, but having it as a table means you can script comparisons across many runs, regression-check in CI, or feed the numbers into spreadsheets and reports.

## Producing Per-Rank Reports

The DDP profiles from `nsys-application.ipynb` were made by wrapping `torchrun` from the outside:

```bash
nsys profile [flags] --output firstOptim torchrun --nproc_per_node=4 ... script.py
```

That produces **one** report containing all four GPUs.  Useful for opening as a single timeline, but multi-report recipes that compare *across ranks* need **one report per rank**.

To produce per-rank reports, flip the nesting: launch `torchrun` from the outside, and put `nsys profile` *inside*, with the rank baked into the output filename.  The standard pattern is a small wrapper script that interpolates `${LOCAL_RANK}` (which `torchrun` sets in each spawned process's environment) into the `--output` flag:

In [ ]:
from pathlib import Path

wrapper_dir = Path("/workspace/scripts")
wrapper_dir.mkdir(parents=True, exist_ok=True)
wrapper = wrapper_dir / "nsys_wrap.sh"

wrapper.write_text("""#!/bin/bash
exec nsys profile \\
  --trace=cuda,nvtx,osrt \\
  --capture-range=cudaProfilerApi \\
  --output=/workspace/reports/firstOptim_rank${LOCAL_RANK} \\
  --force-overwrite=true \\
  python "$@"
""")
wrapper.chmod(0o755)
print(f"Wrote {wrapper}")

Now run the optimized DDP script through `torchrun`, but pass our wrapper as the entrypoint.  `torchrun` invokes the wrapper four times (once per rank), each invocation has `LOCAL_RANK` set to 0..3 in its environment, and the wrapper hands a rank-specific output filename to `nsys profile`.

Note the `--no-python` flag: by default `torchrun <entrypoint>` runs the entrypoint as `python <entrypoint>`, which would feed our bash wrapper to the Python interpreter and fail with a `SyntaxError`.  `--no-python` tells `torchrun` to execute the entrypoint directly instead.

In [ ]:
!cd /workspace/source_code && torchrun --no-python --nproc_per_node=4 --nnodes=1 --standalone --master_addr="localhost" --master_port=1234 /workspace/scripts/nsys_wrap.sh ddp_optimize.py

You should now have four files in `/workspace/reports/`:

In [ ]:
!ls -la /workspace/reports/firstOptim_rank*.nsys-rep

Each of these contains only that rank's view of the training run.  They're smaller and more focused than the all-ranks-in-one report — and, importantly, they're what `nsys recipe`'s multi-report recipes consume.

## Multi-Report Recipe Example

With per-rank files in hand, we can run a recipe that compares across them.  `nccl_gpu_overlap_trace` measures how much of each step's NCCL communication time *overlaps* with GPU compute on the same rank — the higher the overlap, the better-hidden the communication cost.

Note the input form: this recipe's `--input` flag takes *one or more* files (`--input INPUT [INPUT ...]`), so all the per-rank reports go after a **single** `--input`, separated by spaces.  Repeating `--input` for each file does *not* error — argparse simply keeps the last one, and the recipe silently analyses only that single rank.

In [ ]:
!nsys recipe nccl_gpu_overlap_trace \
    --input /workspace/reports/firstOptim_rank0.nsys-rep \
            /workspace/reports/firstOptim_rank1.nsys-rep \
            /workspace/reports/firstOptim_rank2.nsys-rep \
            /workspace/reports/firstOptim_rank3.nsys-rep \
    --output /workspace/reports/overlap_analysis

The recipe writes its analysis to the output directory — typically a CSV plus a small HTML/SVG summary.  Browse `/workspace/reports/overlap_analysis/` for the generated files; the overlap percentages per rank tell you whether NCCL is being well-hidden behind compute or stalling forward progress.

A few other multi-report recipes worth knowing (use `nsys recipe --help` to see the current list on your install):

- **`nccl_gpu_overlap_trace`** — overlap between NCCL communication and GPU compute.
- **`cuda_gpu_kern_sum`** — kernel-level summary across all reports.
- **`gpu_metric_util_map`** — GPU utilization stats per rank.
- **`mpi_gpu_overlap_trace`** — overlap with MPI communication (more relevant in multi-node — see the [multi-node appendix](appendix-multinode.ipynb)).

## When to Use Which Tool

| Question | Best tool |
|---|---|
| "How does this trace differ from the previous one, visually?" | Multi-report view in the GUI |
| "What fraction of step time was in section X?" | Single-report recipe (e.g. `nvtx_sum`) |
| "Is load balanced across ranks?" | Multi-report recipe over per-rank files |
| "Did optimization Y change overlap?" | Multi-report recipe comparing baseline + optimized |
| "Is one specific kernel slow?" | GUI timeline; recipes for aggregate stats |

The GUI is fastest for *looking* at a profile, recipes are fastest for *measuring* one.  In a real performance-tuning workflow you alternate between the two — use the GUI to form a hypothesis, use a recipe to confirm or quantify it.

The next notebook covers Nsight Systems' more advanced tracing features (GPU metrics sampling, NCCL/MPI tracing, multi-process profiling), which compose with everything you've learned here.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red;height:80px;"> <b><br/> [Next Notebook — Advanced Tracing Features](nsight_advanced.ipynb) </b> </div></center>

---
## Links and Resources

- [Nsight Systems multi-report analysis](https://docs.nvidia.com/nsight-systems/UserGuide/index.html#multi-report-analysis)
- [Writing custom nsys recipes](https://docs.nvidia.com/nsight-systems/UserGuide/index.html#nsys-recipe-cli)
- [NVIDIA Nsight Systems documentation](https://docs.nvidia.com/nsight-systems/)

---
## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.